# Twitter Sentiment Analysis (4 Classes)

**How to run in Cursor editor:**
1. Open this notebook in Cursor
2. Select kernel: **Python 3** (top right)
3. Run cells **top to bottom** one by one (Shift+Enter)
4. Or run everything at once: **Run All**

> This notebook shows a simple step-by-step sentiment analysis using Twitter data with 4 classes: positive, negative, neutral, irrelevant

In [ ]:
import os
import re
import string
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from tqdm import tqdm

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# Set working directory
BASE_DIR = Path(r'E:\MachineLearning\Sentiment_Analysis')
os.chdir(BASE_DIR)

print('Working directory:', BASE_DIR)

In [ ]:
# Load Twitter dataset
twitter_data = pd.read_csv(BASE_DIR / 'twitter_sentiment.csv', header=None, names=['tweet_id', 'context', 'sentiment', 'text'])
print('Dataset shape:', twitter_data.shape)

In [ ]:
# Display first few rows
twitter_data.head()

In [ ]:
# Check for missing values
twitter_data.isnull().sum()

In [ ]:
# Remove missing values
twitter_data = twitter_data.dropna()
print('Shape after removing missing values:', twitter_data.shape)

In [ ]:
# Display sentiment distribution
twitter_data['sentiment'].value_counts()

In [ ]:
# Normalize sentiment labels to lowercase
label_map = {
    'Positive': 'positive', 'positive': 'positive',
    'Negative': 'negative', 'negative': 'negative',
    'Neutral': 'neutral', 'neutral': 'neutral',
    'Irrelevant': 'irrelevant', 'irrelevant': 'irrelevant',
}
twitter_data['sentiment'] = twitter_data['sentiment'].map(label_map)

# Keep only valid labels
valid_labels = ['positive', 'negative', 'neutral', 'irrelevant']
twitter_data = twitter_data[twitter_data['sentiment'].isin(valid_labels)]

print('Sentiment distribution after normalization:')
print(twitter_data['sentiment'].value_counts())

In [ ]:
# Visualize sentiment distribution
plt.figure(figsize=(10, 5))
twitter_data['sentiment'].value_counts().plot(kind='bar', color=['#2ecc71', '#e74c3c', '#f39c12', '#95a5a6'])
plt.title('Sentiment Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Text Preprocessing

In [ ]:
# Download NLTK data
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
# Initialize NLTK tools
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [ ]:
# Basic text cleaning
twitter_data['text_clean'] = twitter_data['text'].str.lower()
twitter_data['text_clean'] = twitter_data['text_clean'].str.replace(r'<br\s*/?>', ' ', regex=True)
twitter_data['text_clean'] = twitter_data['text_clean'].str.replace(r'<[^>]+>', ' ', regex=True)
twitter_data['text_clean'] = twitter_data['text_clean'].str.replace(r'http\S+|www\S+', ' ', regex=True)
twitter_data['text_clean'] = twitter_data['text_clean'].str.replace(r'@\w+', ' ', regex=True)
twitter_data['text_clean'] = twitter_data['text_clean'].str.replace(r'\d+', ' ', regex=True)
twitter_data['text_clean'] = twitter_data['text_clean'].str.translate(str.maketrans('', '', string.punctuation))
twitter_data['text_clean'] = twitter_data['text_clean'].str.replace(r'\s+', ' ', regex=True).str.strip()

print('Text cleaning completed')

In [ ]:
# Advanced text preprocessing with NLTK
cleaned_reviews = []

print('Preprocessing text with NLTK... (this takes a few minutes)')
for text in tqdm(twitter_data['text_clean'], desc='Tokenization & Lemmatization'):
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 1]
    cleaned_reviews.append(' '.join(tokens))

twitter_data['text_clean'] = cleaned_reviews
print('NLTK preprocessing completed!')

In [ ]:
# Display original vs cleaned text
twitter_data[['text', 'text_clean', 'sentiment']].head()

In [ ]:
# Remove empty texts after preprocessing
twitter_data = twitter_data[twitter_data['text_clean'].str.len() > 0]
print('Shape after removing empty texts:', twitter_data.shape)

In [ ]:
# Encode labels
label_encoder = LabelEncoder()
twitter_data['label_encoded'] = label_encoder.fit_transform(twitter_data['sentiment'])

print('Label mapping:')
for i, label in enumerate(label_encoder.classes_):
    print(f'{label} -> {i}')

In [ ]:
# Split data into train and test sets
X = twitter_data[['text_clean']]
y = twitter_data['label_encoded']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print('Training set shape:', X_train.shape)
print('Test set shape:', X_test.shape)

In [ ]:
# Create TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=8000, ngram_range=(1, 2), min_df=2, max_df=0.95)

# Fit and transform on training data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train['text_clean'])
X_test_tfidf = tfidf_vectorizer.transform(X_test['text_clean'])

print('TF-IDF features shape - Train:', X_train_tfidf.shape)
print('TF-IDF features shape - Test:', X_test_tfidf.shape)

## Model Training

### Logistic Regression

In [ ]:
# Train Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_tfidf, y_train)

# Make predictions
lr_train_pred = lr_model.predict(X_train_tfidf)
lr_test_pred = lr_model.predict(X_test_tfidf)

# Calculate accuracy
lr_train_acc = accuracy_score(y_train, lr_train_pred)
lr_test_acc = accuracy_score(y_test, lr_test_pred)

print(f'Logistic Regression - Train Accuracy: {lr_train_acc:.4f}')
print(f'Logistic Regression - Test Accuracy: {lr_test_acc:.4f}')

In [ ]:
# Classification report for Logistic Regression
print('Logistic Regression Classification Report:')
print(classification_report(y_test, lr_test_pred, target_names=label_encoder.classes_))

### Random Forest

In [ ]:
# Train Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_tfidf, y_train)

# Make predictions
rf_train_pred = rf_model.predict(X_train_tfidf)
rf_test_pred = rf_model.predict(X_test_tfidf)

# Calculate accuracy
rf_train_acc = accuracy_score(y_train, rf_train_pred)
rf_test_acc = accuracy_score(y_test, rf_test_pred)

print(f'Random Forest - Train Accuracy: {rf_train_acc:.4f}')
print(f'Random Forest - Test Accuracy: {rf_test_acc:.4f}')

In [ ]:
# Classification report for Random Forest
print('Random Forest Classification Report:')
print(classification_report(y_test, rf_test_pred, target_names=label_encoder.classes_))

In [ ]:
# Compare model performance
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Train Accuracy': [lr_train_acc, rf_train_acc],
    'Test Accuracy': [lr_test_acc, rf_test_acc]
})

print('Model Performance Comparison:')
print(results)

In [ ]:
# Visualize model comparison
x = np.arange(len(results))
width = 0.35

plt.figure(figsize=(10, 5))
plt.bar(x - width/2, results['Train Accuracy'], width, label='Train Accuracy')
plt.bar(x + width/2, results['Test Accuracy'], width, label='Test Accuracy')
plt.xticks(x, results['Model'], rotation=15)
plt.title('Model Accuracy Comparison')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Test with custom example
def predict_sentiment(text):
    # Preprocess the text
    text_clean = text.lower()
    text_clean = re.sub(r'<br\s*/?>', ' ', text_clean)
    text_clean = re.sub(r'<[^>]+>', ' ', text_clean)
    text_clean = re.sub(r'http\S+|www\S+', ' ', text_clean)
    text_clean = re.sub(r'@\w+', ' ', text_clean)
    text_clean = re.sub(r'\d+', ' ', text_clean)
    text_clean = text_clean.translate(str.maketrans('', '', string.punctuation))
    text_clean = re.sub(r'\s+', ' ', text_clean).strip()
    
    # Tokenize and lemmatize
    tokens = word_tokenize(text_clean)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 1]
    text_clean = ' '.join(tokens)
    
    # Transform with TF-IDF
    text_tfidf = tfidf_vectorizer.transform([text_clean])
    
    # Predict with Random Forest (best model)
    prediction = rf_model.predict(text_tfidf)
    sentiment = label_encoder.inverse_transform(prediction)[0]
    
    return sentiment

# Test examples
test_texts = [
    "I love this product! It's amazing!",
    "This is terrible, I hate it.",
    "It's okay, nothing special.",
    "This is not relevant to the topic."
]

print("Custom Predictions:")
for text in test_texts:
    sentiment = predict_sentiment(text)
    print(f"Text: {text}")
    print(f"Predicted Sentiment: {sentiment}")
    print()

In [ ]:
joblib.dump(best_model, MODEL_DIR / 'sentiment_model.joblib')
joblib.dump(label_encoder, MODEL_DIR / 'label_encoder.joblib')
joblib.dump(stop_words, MODEL_DIR / 'stop_words.pkl')
joblib.dump(lemmatizer, MODEL_DIR / 'lemmatizer.pkl')
joblib.dump(results_df, MODEL_DIR / 'model_results.pkl')

with open(MODEL_DIR / 'metrics.json', 'w') as f:
    json.dump({
        'best_model': best_model_name,
        'best_test_accuracy': float(best_test_accuracy),
        'all_models': results_df.to_dict(orient='records'),
        'tuning': {
            'Logistic Regression': {'best_cv_accuracy': float(lr_search.best_score_), 'best_params': lr_search.best_params_},
            'Random Forest': {'best_cv_accuracy': float(rf_search.best_score_), 'best_params': rf_search.best_params_},
            'Gradient Boosting': {'best_cv_accuracy': float(gb_search.best_score_), 'best_params': gb_search.best_params_},
            'XGBoost': {'best_cv_accuracy': float(xgb_search.best_score_), 'best_params': xgb_search.best_params_}
        }
    }, f, indent=2)

print('Model saved to models/sentiment_model.joblib')

## Sample Prediction

In [ ]:
input_review = "This movie was absolutely fantastic! Great acting and story."

input_review = input_review.lower()
input_review = re.sub(r'<br\s*/?>', ' ', input_review)
input_review = re.sub(r'<[^>]+>', ' ', input_review)
input_review = re.sub(r'http\S+|www\S+', ' ', input_review)
input_review = re.sub(r'\d+', ' ', input_review)
input_review = input_review.translate(str.maketrans('', '', string.punctuation))
input_review = re.sub(r'\s+', ' ', input_review).strip()

tokens = word_tokenize(input_review)
tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 1]
input_review_clean = ' '.join(tokens)

input_data = pd.DataFrame({'review_clean': [input_review_clean]})

prediction = best_model.predict(input_data)
prediction_proba = best_model.predict_proba(input_data)

print(prediction)
print(prediction_proba)

sentiment_label = label_encoder.inverse_transform(prediction)[0]
print(sentiment_label)